# 01 · Probe library

This notebook defines **every function used in the study**. It is not meant to produce results on its own; the experiment notebooks load it with

```python
exec(compile(load_library_source("01_probe_library.ipynb"), "probe_library", "exec"))
```

(the helper is shown at the top of each experiment notebook). Running this notebook directly executes the self-tests in Section 8, which check the fast implementations against slow reference formulas.

**Contents**

1. Configuration and imports
2. Data container and unsupervised preprocessing
3. Data sanity check (leakage screening)
4. Synthetic data generators with known Bayes AUC
5. Probes: statistical (S1–S4), geometric (G1–G3), model-based (M1–M4)
6. Shared inference layer: permutation tests, Monte Carlo nulls, Simes/Benjamini–Hochberg aggregation
7. Ojala & Garriga (2010) permutation Tests 1 and 2
8. Self-tests

**Notation.** `X` is the feature matrix, `y ∈ {0,1}` the label. Every probe returns a statistic oriented so that **larger values mean stronger evidence of signal**. A probe's p-value is the share of label-permuted (or simulated null) datasets that produce a statistic at least as large as the observed one.

## 1. Configuration and imports

Thread counts are fixed to one per process so that the experiments can be parallelised across processes without oversubscribing the CPU. This must happen before NumPy/scikit-learn are imported.

In [1]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(_v, "1")

import time
import json
import warnings
import importlib.util
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.feature_selection import mutual_info_classif, f_classif
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

ALPHA = 0.05               # significance level used throughout
MAX_N_QUADRATIC = 5000     # row cap for probes whose cost grows with n^2 (S1, S3, S4, G2, G3; G1 for consistency)
MAX_N_MODEL = 10000        # row cap for model-based permutation tests (compute budget, see paper Sec. IV)
CV_FOLDS = 5               # folds for model-based probes
GLOBAL_SEED = 20261011

PROJECT_ROOT = os.path.abspath("..")
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
FIG_DIR = os.path.join(PROJECT_ROOT, "figures")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# problexity (Komorniczak & Ksieniewicz) is loaded from its GitHub source. Only the two
# modules we need are imported, because the package __init__ requires igraph, which is not
# needed for the measures used here.
# The folder is added to sys.path *and* PYTHONPATH so that parallel worker processes can import it too.
import sys
_PX_DIR = os.path.join(PROJECT_ROOT, "ext", "problexity", "problexity", "classification")
if _PX_DIR not in sys.path:
    sys.path.insert(0, _PX_DIR)
os.environ["PYTHONPATH"] = _PX_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")
import feature_based as px_feature      # problexity.classification.feature_based
import neighborhood as px_neigh         # problexity.classification.neighborhood

## 2. Data container and unsupervised preprocessing

`TabularData` stores two views of the same dataset:

* `X` — the **model matrix** used by distance-based and model-based probes: numeric columns median-imputed and standardised; categorical columns mode-imputed and one-hot encoded.
* `Xf` — one column **per original feature** (numeric imputed; categorical as integer codes), used by the per-feature probes S1 and S2.

All preprocessing is **unsupervised**: it never looks at `y`. It can therefore be fitted once on the whole dataset without leaking label information into cross-validation, and it is identical for the observed and every permuted dataset, which keeps the permutation tests exact.

In [2]:
@dataclass
class TabularData:
    name: str
    X: np.ndarray
    y: np.ndarray
    Xf: np.ndarray
    discrete: np.ndarray
    feature_names: list
    meta: dict = field(default_factory=dict)

    @property
    def n(self):
        return len(self.y)


def from_dataframe(df, name, target="y"):
    """Build a TabularData object from a DataFrame with a binary target column."""
    y = df[target].astype(int).to_numpy()
    feats = df.drop(columns=target)
    cat_cols = [c for c in feats.columns if feats[c].dtype == object or str(feats[c].dtype) in ("category", "str", "string")]
    num_cols = [c for c in feats.columns if c not in cat_cols]
    blocks, xf_cols, discrete, groups, col = [], [], [], [], 0
    for c in feats.columns:
        if c in num_cols:
            v = feats[c].astype(float)
            v = v.fillna(v.median()).to_numpy()
            xf_cols.append(v)
            # integer-valued columns with few levels are treated as discrete for the MI estimator
            discrete.append(bool(np.all(np.mod(v, 1) == 0) and len(np.unique(v)) <= 20))
            sd = v.std()
            blocks.append(((v - v.mean()) / (sd if sd > 0 else 1.0))[:, None])
            groups.append(np.array([col])); col += 1
        else:
            v = feats[c].astype(str).where(feats[c].notna(), feats[c].mode().iloc[0])
            codes, uniques = pd.factorize(v)
            xf_cols.append(codes.astype(float))
            discrete.append(True)
            blocks.append(np.eye(len(uniques))[codes])
            groups.append(np.arange(col, col + len(uniques))); col += len(uniques)
    X = np.hstack(blocks).astype(float)
    Xf = np.column_stack(xf_cols).astype(float)
    return TabularData(name, X, y, Xf, np.array(discrete), list(feats.columns),
                       {"n_categorical": len(cat_cols), "n_numeric": len(num_cols), "groups": groups})


def from_arrays(X, y, name, meta=None):
    """Build a TabularData object from a numeric array (used for synthetic data)."""
    X = np.asarray(X, float)
    return TabularData(name, X, np.asarray(y, int), X.copy(), np.zeros(X.shape[1], bool),
                       [f"x{j}" for j in range(X.shape[1])], meta or {})


def load_processed(name):
    """Read a processed dataset written by notebook 00."""
    return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"))


def subsample(data, max_n, rng):
    """Stratified random subsample of at most max_n rows (returns the object unchanged if smaller)."""
    if data.n <= max_n:
        return data
    idx = []
    for c in (0, 1):
        cls = np.flatnonzero(data.y == c)
        k = int(round(max_n * len(cls) / data.n))
        idx.append(rng.choice(cls, size=k, replace=False))
    idx = np.sort(np.concatenate(idx))
    return TabularData(data.name, data.X[idx], data.y[idx], data.Xf[idx], data.discrete,
                       data.feature_names, dict(data.meta, subsampled_from=data.n))

## 3. Data sanity check (leakage screening)

Leakage produces apparent signal that every probe will happily detect (Kaufman et al., 2012; Kapoor & Narayanan, 2023). Before any probe runs, `sanity_check` screens for three common patterns:

1. **Identifier-like columns**: integer or string columns whose values are almost unique per row (> 95 % distinct).
2. **Exact duplicate rows** (features and label).
3. **Single-feature near-perfect prediction**: any one feature that alone reaches AUC > 0.99. Numeric features are scored by their raw values; categorical features by an out-of-fold target encoding.

These checks catch crude leakage only. Subtle leakage, such as an outcome measured after the event of interest, can look like an ordinary feature; notebook 06 demonstrates this with the NSW `re78` column.

In [3]:
def single_feature_auc(df, target="y", seed=0):
    """AUC of each feature used on its own (max(AUC, 1-AUC)); categorical features use out-of-fold target means."""
    y = df[target].to_numpy()
    out = {}
    for c in df.columns.drop(target):
        v = df[c]
        if v.dtype == object or str(v.dtype) in ("category", "str", "string"):
            v = v.astype(str)
            enc = np.zeros(len(v))
            for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(v, y):
                means = pd.Series(y[tr]).groupby(v.iloc[tr].to_numpy()).mean()
                enc[te] = v.iloc[te].map(means).fillna(y[tr].mean()).to_numpy()
            score = enc
        else:
            score = v.astype(float).fillna(v.median()).to_numpy()
        if np.unique(score).size < 2:
            out[c] = 0.5
            continue
        a = roc_auc_score(y, score)
        out[c] = max(a, 1 - a)
    return pd.Series(out).sort_values(ascending=False)


def sanity_check(df, target="y", id_unique_ratio=0.95, auc_flag=0.99):
    """Screen a DataFrame for crude leakage. Returns a dict with findings and a verdict."""
    n = len(df)
    id_like = []
    for c in df.columns.drop(target):
        v = df[c].dropna()
        is_int_or_str = v.dtype == object or str(v.dtype) in ("str", "string") or np.all(np.mod(v.astype(float), 1) == 0)
        if is_int_or_str and v.nunique() / n > id_unique_ratio:
            id_like.append(c)
    n_dup = int(df.duplicated().sum())
    sfa = single_feature_auc(df, target)
    flagged = list(sfa[sfa > auc_flag].index)
    leak = bool(id_like or flagged)
    return {"id_like_columns": id_like, "exact_duplicates": n_dup, "duplicate_rate": n_dup / n,
            "max_single_feature_auc": float(sfa.iloc[0]), "best_single_feature": sfa.index[0],
            "flagged_features": flagged, "verdict": "leakage suspected" if leak else "clean"}

## 4. Synthetic data generators with known Bayes AUC

Features are independent standard normals. The label is drawn from a logistic model

$$P(Y=1\mid x) = \sigma\big(s\cdot g(x) + b\big),$$

where $g$ defines the **signal type**, $s \ge 0$ the **signal strength** and $b$ the **class balance**. Because $P(Y=1\mid x)$ is known, the Bayes-optimal score is $g(x)$ itself and the **Bayes AUC** can be computed exactly up to Monte Carlo error.

| Type | $g(x)$ | Visible to single-feature tests? |
|---|---|---|
| `null` | $0$ (Y independent of X) | – |
| `linear` | $\sum_{j=1}^{5} x_j/\sqrt5$ | yes (mean shifts) |
| `nonlinear` | $(\sum_{j=1}^{5} x_j^2 - 5)/\sqrt{10}$ | partly: classes differ in spread, not in mean |
| `interaction` | $x_1 x_2$ | no: each feature alone is independent of $Y$ when classes are balanced |

The remaining `n_noise` columns are irrelevant. `calibrate_generator` finds $(s, b)$ so that the population hits a target Bayes AUC and positive rate.

In [4]:
N_INFORMATIVE = 5

def signal_score(kind, X):
    """Bayes-optimal score g(x) for each signal type."""
    if kind == "null":
        return np.zeros(len(X))
    if kind == "linear":
        return X[:, :N_INFORMATIVE].sum(1) / np.sqrt(N_INFORMATIVE)
    if kind == "nonlinear":
        return ((X[:, :N_INFORMATIVE] ** 2).sum(1) - N_INFORMATIVE) / np.sqrt(2 * N_INFORMATIVE)
    if kind == "interaction":
        return X[:, 0] * X[:, 1]
    raise ValueError(kind)


def _sigmoid(z):
    return 1 / (1 + np.exp(-z))


def population_auc(score, p):
    """Exact AUC of a score when Y~Bernoulli(p): weighted AUC without sampling Y."""
    s = np.concatenate([score, score])
    lab = np.concatenate([np.ones_like(score), np.zeros_like(score)])
    w = np.concatenate([p, 1 - p])
    return roc_auc_score(lab, s, sample_weight=w)


_CALIBRATION_CACHE = {}
_CALIBRATION_FILE = os.path.join(RESULTS_DIR, "generator_calibration.json")   # shared by worker processes

def calibrate_generator(kind, target_auc, pos_rate=0.5, n_mc=200_000, seed=12345):
    """Find (s, b) such that the Bayes AUC ≈ target_auc and P(Y=1) ≈ pos_rate (nested bisection).

    Results are cached in memory and in a JSON file so that parallel worker processes do not repeat
    the (deterministic) calibration.
    """
    key = (kind, round(target_auc, 4), round(pos_rate, 4))
    skey = "|".join(map(str, key))
    if key in _CALIBRATION_CACHE:
        return _CALIBRATION_CACHE[key]
    if os.path.exists(_CALIBRATION_FILE):
        try:
            stored = json.load(open(_CALIBRATION_FILE))
            if skey in stored:
                _CALIBRATION_CACHE[key] = stored[skey]
                return stored[skey]
        except (json.JSONDecodeError, OSError):
            pass
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n_mc, N_INFORMATIVE))
    g = signal_score(kind, X)

    def fit_b(s):
        lo, hi = -30.0, 30.0
        for _ in range(60):
            mid = (lo + hi) / 2
            if _sigmoid(s * g + mid).mean() < pos_rate:
                lo = mid
            else:
                hi = mid
        return (lo + hi) / 2

    lo, hi = 0.0, 20.0
    for _ in range(40):
        s = (lo + hi) / 2
        b = fit_b(s)
        if population_auc(g, _sigmoid(s * g + b)) < target_auc:
            lo = s
        else:
            hi = s
    s = (lo + hi) / 2
    b = fit_b(s)
    out = {"s": s, "b": b, "bayes_auc": population_auc(g, _sigmoid(s * g + b))}
    _CALIBRATION_CACHE[key] = out
    try:
        stored = json.load(open(_CALIBRATION_FILE)) if os.path.exists(_CALIBRATION_FILE) else {}
    except (json.JSONDecodeError, OSError):
        stored = {}
    stored[skey] = out
    tmp = _CALIBRATION_FILE + f".{os.getpid()}.tmp"
    json.dump(stored, open(tmp, "w"), indent=1)
    os.replace(tmp, _CALIBRATION_FILE)
    return out


def make_synthetic(kind, n, target_auc=0.7, n_noise=20, pos_rate=0.5, seed=0):
    """Draw one synthetic dataset. Returns TabularData with meta['bayes_auc']."""
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, N_INFORMATIVE + n_noise))
    if kind == "null":
        y = (rng.random(n) < pos_rate).astype(int)
        meta = {"kind": kind, "bayes_auc": 0.5, "pos_rate": pos_rate, "n_noise": n_noise}
    else:
        cal = calibrate_generator(kind, target_auc, pos_rate)
        p = _sigmoid(cal["s"] * signal_score(kind, X) + cal["b"])
        y = (rng.random(n) < p).astype(int)
        meta = {"kind": kind, "bayes_auc": cal["bayes_auc"], "pos_rate": pos_rate, "n_noise": n_noise}
    if y.min() == y.max():                      # guard against a single-class draw at tiny n
        y[rng.integers(n)] = 1 - y[0]
    return from_arrays(X, y, f"{kind}_n{n}", meta)

## 5. Probes

Every probe has two methods:

* `prepare(data)` — label-independent pre-computation (distance matrices, minimum spanning trees, …). It is done **once** per dataset, and then reused for every permutation, which makes permutation tests of the model-free probes cheap.
* `stat(state, y)` — the statistic for a given label vector (larger = stronger evidence of signal).

Per-feature probes (S1, S2) return a **vector** (one value per original feature); the inference layer turns it into per-feature p-values and a dataset-level decision.

### 5.1 Statistical probes

**S1 – Mutual information.** scikit-learn's `mutual_info_classif`, which uses the k-nearest-neighbour estimator of Ross (2014) for continuous features and a contingency estimate for discrete ones. Per-feature p-values come from permutations or a Monte Carlo null.

**S2 – ANOVA F / chi-square.** Classical parametric tests: ANOVA F for numeric features, Pearson chi-square test of independence for categorical features. These have their own analytic p-values, so no permutation is needed.

**S3 – Distance correlation.** For binary $y$, with $A$ the double-centred Euclidean distance matrix of $X$, the V-statistic simplifies to a quadratic form,
$$\mathrm{dCov}_n^2(X,Y)=\tfrac{1}{n^2}\sum_{ij}A_{ij}\,|y_i-y_j| = -\tfrac{2}{n^2}\,y^\top A\,y,$$
because the rows of $A$ sum to zero. $\mathrm{dCor}=\sqrt{\mathrm{dCov}^2/\sqrt{\mathrm{dVar}^2(X)\,\mathrm{dVar}^2(Y)}}$. Pre-computing $A$ once makes each permutation a single matrix–vector product.

**S4 – HSIC.** Gaussian kernel on $X$ (bandwidth = median pairwise distance) and the delta kernel $L_{ij}=1[y_i=y_j]$ on $Y$. With $\tilde K = HKH$ (centred kernel), $\mathrm{HSIC}_b = \tfrac{1}{n^2}\mathrm{tr}(\tilde K L) = \tfrac{2}{n^2}\,y^\top \tilde K y$. For a binary label, HSIC with a Gaussian kernel and dCor are kernel and distance versions of the same two-sample comparison of the class-conditional distributions (Sejdinovic et al., 2013).

In [5]:
class Probe:
    """Base class. Subclasses set name, family, per_feature and implement prepare/stat."""
    name, family, per_feature, max_n = "base", "", False, None

    def prepare(self, data):
        return data

    def stat(self, state, y):
        raise NotImplementedError


class MutualInformationProbe(Probe):
    name, family, per_feature, max_n = "S1_MI", "statistical", True, MAX_N_QUADRATIC

    def stat(self, data, y):
        return mutual_info_classif(data.Xf, y, discrete_features=data.discrete, n_neighbors=3, random_state=0)


class AnovaChi2Probe(Probe):
    """Per-feature parametric p-values (returned directly; see `parametric=True`)."""
    name, family, per_feature, max_n = "S2_ANOVA_Chi2", "statistical", True, None
    parametric = True

    def pvalues(self, data, y):
        p = np.ones(data.Xf.shape[1])
        for j in range(data.Xf.shape[1]):
            col = data.Xf[:, j]
            if np.unique(col).size < 2:
                continue
            if data.discrete[j] and np.unique(col).size <= 50:
                table = pd.crosstab(col, y).to_numpy()
                if table.shape[0] > 1 and table.shape[1] > 1:
                    p[j] = stats.chi2_contingency(table)[1]
            else:
                p[j] = f_classif(col[:, None], y)[1][0]
        return np.nan_to_num(p, nan=1.0)


def _double_centre(M):
    return M - M.mean(0, keepdims=True) - M.mean(1, keepdims=True) + M.mean()


class DistanceCorrelationProbe(Probe):
    name, family, max_n = "S3_dCor", "statistical", MAX_N_QUADRATIC

    def prepare(self, data):
        A = _double_centre(squareform(pdist(data.X)))
        return {"A": A, "dvar_x": (A ** 2).mean(), "n": data.n}

    def stat(self, st, y):
        y = y.astype(float)
        n = st["n"]
        dcov2 = -2.0 * y @ (st["A"] @ y) / n ** 2
        # dVar(Y) depends only on n and the number of positives, so it is identical for every
        # permutation and is computed once and cached.
        key = int(y.sum())
        if key not in st.setdefault("dvar_y", {}):
            B = _double_centre(np.abs(y[:, None] - y[None, :]))
            st["dvar_y"][key] = (B ** 2).mean()
        return float(np.sqrt(max(dcov2, 0) / np.sqrt(st["dvar_x"] * st["dvar_y"][key])))


class HSICProbe(Probe):
    name, family, max_n = "S4_HSIC", "statistical", MAX_N_QUADRATIC

    def prepare(self, data):
        D = squareform(pdist(data.X))
        sigma = np.median(D[np.triu_indices_from(D, 1)])
        sigma = sigma if sigma > 0 else 1.0
        Kc = _double_centre(np.exp(-D ** 2 / (2 * sigma ** 2)))
        return {"Kc": Kc, "n": data.n}

    def stat(self, st, y):
        y = y.astype(float)
        return float(2.0 * y @ (st["Kc"] @ y) / st["n"] ** 2)

### 5.2 Geometric probes (data-complexity measures, Lorena et al., 2019)

* **G1 – Maximum Fisher's discriminant ratio (F1).** Computed with `problexity.f1`, which returns $1/(1+\max_j r_j)$ (higher = harder). The probe uses $1-F1$ so that larger means more signal.
* **G2 – Fraction of borderline points (N1).** Build the minimum spanning tree (MST) of the data once; N1 is the share of points joined by an MST edge to a point of the other class (Lorena et al.'s definition). The probe uses $-N1$. A tiny constant is added to all off-diagonal distances so that duplicate rows (distance 0) remain connected in the MST.
* **G3 – Leave-one-out 1-NN error (N3).** The nearest neighbour of each point is found once; N3 is the share of points whose nearest neighbour has the other label. The probe uses $-N3$.

Section 8 checks G2/G3 against problexity's own implementations.

In [6]:
class FisherRatioProbe(Probe):
    name, family, max_n = "G1_F1", "geometric", MAX_N_QUADRATIC

    def stat(self, data, y):
        with np.errstate(divide="ignore", invalid="ignore"):   # constant columns give 0/0 ratios, ignored by nanmax
            return float(1.0 - px_feature.f1(data.X, y))


def mst_edges(X):
    """Edges (i, j) of the Euclidean minimum spanning tree."""
    D = squareform(pdist(X))
    D[D == 0] = 1e-12
    np.fill_diagonal(D, 0)
    T = minimum_spanning_tree(D).tocoo()
    return np.column_stack([T.row, T.col])


class BorderlineProbe(Probe):
    name, family, max_n = "G2_N1", "geometric", MAX_N_QUADRATIC

    def prepare(self, data):
        return {"edges": mst_edges(data.X), "n": data.n}

    def stat(self, st, y):
        e = st["edges"]
        cross = y[e[:, 0]] != y[e[:, 1]]
        border = np.zeros(st["n"], bool)
        border[e[cross, 0]] = True
        border[e[cross, 1]] = True
        return float(-border.mean())


class NearestNeighbourErrorProbe(Probe):
    name, family, max_n = "G3_N3", "geometric", MAX_N_QUADRATIC

    def prepare(self, data):
        nn = NearestNeighbors(n_neighbors=2).fit(data.X)
        idx = nn.kneighbors(data.X, return_distance=False)
        # column 0 is normally the point itself; with exact duplicates it may be a twin, so take the
        # first neighbour that is not the point itself
        own = np.arange(data.n)
        first = np.where(idx[:, 0] != own, idx[:, 0], idx[:, 1])
        return {"nn": first}

    def stat(self, st, y):
        return float(-(y != y[st["nn"]]).mean())

### 5.3 Model-based probes

Each model-based probe's statistic is the **out-of-fold AUC** from stratified 5-fold cross-validation (predictions pooled over folds). The dummy baseline has AUC 0.5 by construction, so the effect size is the **AUC gain** = AUC − 0.5.

| ID | Model | Settings |
|---|---|---|
| M1 | Shallow decision tree | `max_depth=3` |
| M2 | L2 logistic regression | `C=1`, standardised inputs |
| M3 | Random forest | 50 trees, `min_samples_leaf=5` (flexible learner; replaces TabPFN, which could not be installed, see paper) |
| M4 | Histogram gradient boosting | scikit-learn defaults (LightGBM-style algorithm; reference probe) |

In [7]:
def make_model(key, seed=0):
    if key == "tree":
        return DecisionTreeClassifier(max_depth=3, random_state=seed)
    if key == "logreg":
        return LogisticRegression(C=1.0, max_iter=2000)
    if key == "forest":
        return RandomForestClassifier(n_estimators=50, min_samples_leaf=5, n_jobs=1, random_state=seed)
    if key == "hgb":
        return HistGradientBoostingClassifier(random_state=seed)
    raise ValueError(key)


def cv_auc(model_key, X, y, seed=0, folds=CV_FOLDS):
    """Out-of-fold AUC of a model under stratified k-fold CV."""
    cv = StratifiedKFold(folds, shuffle=True, random_state=seed)
    p = cross_val_predict(make_model(model_key, seed), X, y, cv=cv, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, p))


class ModelProbe(Probe):
    family, max_n = "model-based", MAX_N_MODEL

    def __init__(self, model_key, name):
        self.model_key, self.name = model_key, name

    def stat(self, data, y):
        return cv_auc(self.model_key, data.X, y)


def all_probes():
    """The probe panel compared in this study, in a fixed order."""
    return [MutualInformationProbe(), AnovaChi2Probe(), DistanceCorrelationProbe(), HSICProbe(),
            FisherRatioProbe(), BorderlineProbe(), NearestNeighbourErrorProbe(),
            ModelProbe("tree", "M1_Tree"), ModelProbe("logreg", "M2_LogReg"),
            ModelProbe("forest", "M3_Forest"), ModelProbe("hgb", "M4_HGB")]

PROBE_ORDER = [p.name for p in all_probes()]
FAMILY = {p.name: p.family for p in all_probes()}

## 6. Shared inference layer

**Permutation p-value** (Ojala & Garriga, 2010; Phipson & Smyth, 2010):
$$p = \frac{1 + \#\{b: T_b \ge T_{\text{obs}}\}}{B+1},$$
which is never zero and gives an exact level-α test when rows are exchangeable (Hemerik & Goeman, 2018).

**Early stopping.** Once $1+\#\{T_b \ge T_{\text{obs}}\} > \alpha (B+1)$, the final p-value must exceed α, so the run can stop without changing the decision. Early-stopped runs report `p_lower_bound`.

**Monte Carlo null** (synthetic experiments). When the data-generating process is known, the null distribution of a statistic for a given setting (n, number of features, class balance) is simulated once from datasets with $Y$ independent of $X$; every replicate in that setting is then compared with this shared null with the same formula. This is the same test in distribution and saves roughly two orders of magnitude of compute.

**Per-feature aggregation.** For S1/S2 the dataset-level p-value is the **Simes** combination $p_{\text{Simes}}=\min_k\, m\,p_{(k)}/k$; rejecting when $p_{\text{Simes}}\le\alpha$ is identical to the Benjamini–Hochberg procedure rejecting at least one feature. The **uncorrected** rule (reject if any raw $p_j \le \alpha$) is also recorded to measure the inflation it causes.

In [8]:
def simes(p):
    """Simes global p-value (= BH 'any rejection' decision)."""
    p = np.sort(np.asarray(p, float))
    m = len(p)
    return float(min(1.0, np.min(p * m / np.arange(1, m + 1))))


def bh_reject(p, alpha=ALPHA):
    """Boolean mask of features rejected by Benjamini–Hochberg at level alpha."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    m = len(p)
    thresh = alpha * np.arange(1, m + 1) / m
    passed = p[order] <= thresh
    k = np.max(np.nonzero(passed)[0]) + 1 if passed.any() else 0
    mask = np.zeros(m, bool)
    mask[order[:k]] = True
    return mask


def mc_pvalue(t_obs, null_values):
    """Monte Carlo / permutation p-value with the +1 correction."""
    null_values = np.asarray(null_values, float)
    return float((1 + np.sum(null_values >= t_obs - 1e-12)) / (len(null_values) + 1))


def run_probe(probe, data, B=200, seed=0, alpha=ALPHA, early_stop=False):
    """Observed statistic and permutation p-value of one probe on one dataset.

    Returns a dict with: probe, statistic, p (dataset level), p_uncorrected_min (per-feature probes),
    n_used, B_done, early_stopped, seconds_observed, seconds_total.
    """
    rng = np.random.default_rng(seed)
    if probe.max_n is not None and data.n > probe.max_n:
        data = subsample(data, probe.max_n, rng)
    y = data.y
    t0 = time.perf_counter()
    if getattr(probe, "parametric", False):
        pv = probe.pvalues(data, y)
        sec = time.perf_counter() - t0
        return {"probe": probe.name, "statistic": float(-np.log10(max(simes(pv), 1e-300))), "p": simes(pv),
                "p_uncorrected_min": float(pv.min()), "n_features_rejected_bh": int(bh_reject(pv, alpha).sum()),
                "n_used": data.n, "B_done": 0, "early_stopped": False,
                "seconds_observed": sec, "seconds_total": sec}
    st = probe.prepare(data)
    t_obs = probe.stat(st, y)
    sec_obs = time.perf_counter() - t0
    if probe.per_feature:
        t_obs = np.asarray(t_obs, float)
        exceed = np.zeros_like(t_obs)
    else:
        exceed = 0
    b_done, stopped = 0, False
    for b in range(B):
        tb = probe.stat(st, rng.permutation(y))
        exceed = exceed + (np.asarray(tb) >= np.asarray(t_obs) - 1e-12)
        b_done += 1
        if early_stop and not probe.per_feature and (1 + exceed) > alpha * (B + 1):
            stopped = True
            break
    total = time.perf_counter() - t0
    if probe.per_feature:
        pv = (1 + exceed) / (b_done + 1)
        return {"probe": probe.name, "statistic": float(np.max(t_obs)), "p": simes(pv),
                "p_uncorrected_min": float(pv.min()), "n_features_rejected_bh": int(bh_reject(pv, alpha).sum()),
                "n_used": data.n, "B_done": b_done, "early_stopped": False,
                "seconds_observed": sec_obs, "seconds_total": total}
    p = (1 + exceed) / (b_done + 1) if not stopped else (1 + exceed) / (B + 1)
    return {"probe": probe.name, "statistic": float(t_obs), "p": float(p), "p_uncorrected_min": np.nan,
            "n_features_rejected_bh": np.nan, "n_used": data.n, "B_done": b_done, "early_stopped": stopped,
            "p_lower_bound": bool(stopped), "seconds_observed": sec_obs, "seconds_total": total}


def observed_statistics(data, probes=None, seed=0):
    """Observed statistic of every probe on one dataset (no permutations), with timing.

    Per-feature probes return their vector of statistics (S1) or p-values (S2). Used with Monte Carlo nulls.
    """
    rng = np.random.default_rng(seed)
    out = {}
    for probe in probes or all_probes():
        d = subsample(data, probe.max_n, rng) if probe.max_n and data.n > probe.max_n else data
        t0 = time.perf_counter()
        if getattr(probe, "parametric", False):
            val = probe.pvalues(d, d.y)
        else:
            val = probe.stat(probe.prepare(d), d.y)
        out[probe.name] = (np.asarray(val, float) if np.ndim(val) else float(val), time.perf_counter() - t0)
    return out


def repeated_cv_gain(model_key, data, repeats=5, seed=0):
    """AUC gain over the dummy baseline from repeated 5-fold CV: mean and 2.5/97.5 percentiles over repeats."""
    g = np.array([cv_auc(model_key, data.X, data.y, seed=seed + r) - 0.5 for r in range(repeats)])
    return {"gain_mean": g.mean(), "gain_lo": np.percentile(g, 2.5), "gain_hi": np.percentile(g, 97.5)}


def two_stage_decision(p_stage1, p_stage2, alpha=ALPHA):
    """Two-stage protocol: reject H0 if either stage is significant at alpha/2 (Bonferroni)."""
    return bool(min(p_stage1, p_stage2) <= alpha / 2)

## 7. Ojala & Garriga (2010): permutation Tests 1 and 2

Ojala and Garriga proposed two permutation tests for a classifier's cross-validated error $e(f, D)$:

* **Test 1 — label permutation.** $H_0$: features and labels are independent. Permute the labels, keep $X$. A small p-value means the classifier has found real class structure.
* **Test 2 — within-class feature permutation.** $H_0$: given the class, the features are independent of each other, so the classifier gains nothing from dependencies between features. Permute each feature's values **independently within each class**. This keeps every class-conditional marginal distribution but destroys the dependency structure. A small p-value means the classifier *exploits dependencies between features*, e.g. an interaction.

p-values follow their definition: $p = \big(\#\{D' : e(f,D') \le e(f,D)\} + 1\big)/(k+1)$. As in the original paper, the error is estimated with 10-fold cross-validation, and the same folds are used for the original and every permuted dataset.

In [9]:
def ojala_classifier(key, seed=0):
    """Classifiers used in the Ojala–Garriga replication."""
    if key == "decision_tree":
        return DecisionTreeClassifier(random_state=seed)
    if key == "naive_bayes":
        return GaussianNB()
    if key == "1nn":
        return KNeighborsClassifier(n_neighbors=1)
    if key == "svm_rbf":
        return SVC(kernel="rbf", C=1.0, gamma="scale")
    raise ValueError(key)


def cv_error(clf_key, X, y, folds, seed=0):
    """10-fold CV error rate (share of misclassified points) on fixed folds."""
    wrong = 0
    for tr, te in folds:
        m = ojala_classifier(clf_key, seed).fit(X[tr], y[tr])
        wrong += np.sum(m.predict(X[te]) != y[te])
    return wrong / len(y)


def permute_within_class(X, y, rng, groups=None):
    """Ojala–Garriga Test 2 permutation: shuffle each feature independently inside each class.

    `groups` lists the model-matrix columns that belong to one original feature (e.g. the one-hot
    columns of a categorical variable); a group is shuffled as a block so that every row keeps a valid
    one-hot code. Without groups, every column is its own feature.
    """
    groups = groups if groups is not None else [np.array([j]) for j in range(X.shape[1])]
    Xp = X.copy()
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        for g in groups:
            Xp[np.ix_(idx, g)] = X[np.ix_(rng.permutation(idx), g)]
    return Xp


def ojala_garriga_tests(clf_key, X, y, k=200, n_folds=10, seed=0, groups=None):
    """Run Test 1 and Test 2 for one classifier on one dataset.

    Returns observed error, p-values and mean permuted errors for both tests.
    """
    rng = np.random.default_rng(seed)
    folds = list(StratifiedKFold(n_folds, shuffle=True, random_state=seed).split(X, y))
    e_obs = cv_error(clf_key, X, y, folds, seed)
    e1 = np.array([cv_error(clf_key, X, rng.permutation(y), folds, seed) for _ in range(k)])
    e2 = np.array([cv_error(clf_key, permute_within_class(X, y, rng, groups), y, folds, seed) for _ in range(k)])
    return {"classifier": clf_key, "error": e_obs,
            "p_test1": (np.sum(e1 <= e_obs) + 1) / (k + 1), "mean_error_test1": e1.mean(),
            "p_test2": (np.sum(e2 <= e_obs) + 1) / (k + 1), "mean_error_test2": e2.mean(), "k": k}

## 8. Self-tests

These checks run when the notebook is executed directly (they are skipped when the library is loaded from another notebook). They compare the fast implementations with slow textbook formulas and with problexity.

In [10]:
def _dcor_reference(X, y):
    """Textbook distance correlation (Székely & Rizzo) for checking the fast version."""
    a = _double_centre(squareform(pdist(X)))
    b = _double_centre(squareform(pdist(y.reshape(-1, 1).astype(float))))
    dcov2, dvx, dvy = (a * b).mean(), (a * a).mean(), (b * b).mean()
    return np.sqrt(dcov2 / np.sqrt(dvx * dvy))


def _hsic_reference(X, y):
    """Textbook biased HSIC with delta kernel on y."""
    n = len(y)
    D = squareform(pdist(X))
    sigma = np.median(D[np.triu_indices_from(D, 1)])
    K = np.exp(-D ** 2 / (2 * sigma ** 2))
    L = (y[:, None] == y[None, :]).astype(float)
    H = np.eye(n) - 1.0 / n
    return np.trace(K @ H @ L @ H) / n ** 2


def run_self_tests():
    rng = np.random.default_rng(1)
    d = make_synthetic("linear", 300, 0.75, n_noise=5, seed=3)
    # distance correlation and HSIC
    st = DistanceCorrelationProbe().prepare(d)
    assert np.isclose(DistanceCorrelationProbe().stat(st, d.y), _dcor_reference(d.X, d.y))
    st = HSICProbe().prepare(d)
    assert np.isclose(HSICProbe().stat(st, d.y), _hsic_reference(d.X, d.y))
    # N1: our vertex-based definition vs problexity (which reports cross edges / 2n)
    e = mst_edges(d.X)
    cross_edges = np.sum(d.y[e[:, 0]] != d.y[e[:, 1]])
    assert np.isclose(cross_edges / 2 / d.n, px_neigh.n1(d.X, d.y))
    # N3 equals problexity's leave-one-out 1-NN error
    st = NearestNeighbourErrorProbe().prepare(d)
    assert np.isclose(-NearestNeighbourErrorProbe().stat(st, d.y), px_neigh.n3(d.X, d.y))
    # Simes equals BH any-rejection
    for _ in range(200):
        p = rng.random(10) ** 3
        assert (simes(p) <= ALPHA) == bh_reject(p).any()
    # generator calibration hits its target
    cal = calibrate_generator("interaction", 0.7)
    assert abs(cal["bayes_auc"] - 0.7) < 0.005
    print("All self-tests passed.")

if __name__ == "__main__" and not globals().get("_LIBRARY_IMPORT", False):
    run_self_tests()

All self-tests passed.
